In [34]:
import pandas as pd
import numpy as np
import os
import glob
from cleaning_script import adaptive_clean_real_estate_data

In [3]:
#GOING BACK AND LOOKING AT THE RAW DATASETS
data_dir = '../original_datasets'

# Pattern to match all CRMLS monthly files
pattern = os.path.join(data_dir, "CRMLSSold2025??_filled.csv")

# Get all matching file paths
file_list = sorted(glob.glob(pattern))

print("Files found:", file_list)

# Load and store each DataFrame
dfs = []
for f in file_list:
    df = pd.read_csv(f, low_memory=False)
    df["SourceFile"] = os.path.basename(f)  # optional: track the month
    dfs.append(df)

# Merge vertically (stack rows)
merged_df = pd.concat(dfs, ignore_index=True)

print("Merged shape:", merged_df.shape)



Files found: ['../original_datasets\\CRMLSSold202502_filled.csv', '../original_datasets\\CRMLSSold202503_filled.csv', '../original_datasets\\CRMLSSold202504_filled.csv', '../original_datasets\\CRMLSSold202505_filled.csv', '../original_datasets\\CRMLSSold202506_filled.csv', '../original_datasets\\CRMLSSold202507_filled.csv']
Merged shape: (133092, 81)


In [23]:
aug = pd.read_csv('../original_datasets/CRMLSSold202508_filled-2.csv')
sept = pd.read_csv('../original_datasets/CRMLSSold202509.csv')
merged_df

,BuyerAgentAOR,ListAgentAOR,Flooring,ViewYN,WaterfrontYN,BasementYN,PoolPrivateYN,OriginalListPrice,ListingKey,ListAgentEmail,...,NewConstructionYN,GarageSpaces,HighSchoolDistrict,PostalCode,AssociationFee,LotSizeSquareFeet,MiddleOrJuniorSchoolDistrict,latfilled,lonfilled,SourceFile
2,SanDiego,SanDiego,NaN,False,NaN,NaN,False,880000.0,497696903,lenskab@gmail.com,...,False,2.0,NaN,91942,NaN,NaN,NaN,False,False,CRMLSSold202502_filled.csv
3,SanDiego,SanDiego,NaN,False,NaN,NaN,False,875000.0,497696407,lenskab@gmail.com,...,False,2.0,NaN,91942,NaN,NaN,NaN,False,False,CRMLSSold202502_filled.csv
4,SanDiego,SanDiego,NaN,False,NaN,NaN,False,849000.0,486616176,lenskab@gmail.com,...,False,2.0,NaN,91942,NaN,NaN,NaN,False,False,CRMLSSold202502_filled.csv
15,SouthBay,SouthBay,Carpet,True,NaN,True,False,1100000.0,1108119618,elaine@elainemallon.com,...,False,2.0,Los Angeles Unified,90731,0.0,4501.0,NaN,False,False,CRMLSSold202502_filled.csv
16,ContraCosta,ContraCosta,Carpet,NaN,NaN,NaN,False,760000.0,1108119338,sabine.larsen@redfin.com,...,False,2.0,NaN,94579,NaN,5040.0,NaN,False,False,CRMLSSold202502_filled.csv
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
156022,NorthSanDiegoCounty,NorthSanDiegoCounty,NaN,True,NaN,NaN,False,5999000.0,1053100390,CARYNGILDEA@AOL.COM,...,NaN,3.0,Bonsall Unified,92084,0.0,1742400.0,NaN,False,False,NaN
156023,HighDesert,HighDesert,"Bamboo,Carpet",True,NaN,NaN,False,11500000.0,1052425061,annamwoods@gmail.com,...,False,2.5,Bear Valley Unified,92314,0.0,219024.0,NaN,False,False,NaN
156038,InlandValleys,InlandValleys,NaN,True,NaN,NaN,False,410000.0,1044748983,miles@milesturnerhomes.com,...,True,2.0,Other,92884,284.0,3754.0,NaN,False,False,NaN
156047,NorthSanDiegoCounty,NorthSanDiegoCounty,NaN,True,NaN,NaN,False,1399900.0,1038358314,dfontana@calwestliving.com,...,True,2.0,Escondido Union,92026,521.0,87120.0,NaN,False,False,NaN


In [24]:
merged_df = pd.concat([merged_df, aug, sept], ignore_index=True)
merged_df.shape

(123802, 81)

In [25]:
merged_df = merged_df[(merged_df['PropertyType']=='Residential') & (merged_df['PropertySubType']=='SingleFamilyResidence')]


In [35]:
testing_set = pd.read_csv('../original_datasets/CRMLSSold202510.csv')

In [36]:
train_cleaned, stats, mechs = adaptive_clean_real_estate_data(merged_df, fit = True)
test_clean, _, _ = adaptive_clean_real_estate_data(testing_set, fit = False, imputation_stats=stats, missing_mechanisms=mechs, trim_outliers=True )

ADAPTIVE CLEANING - TRAINING DATA
Initial shape: (101297, 81)

Dropping 21 columns with >75% missing

DIAGNOSING MISSING DATA MECHANISMS

ViewYN: NMAR (amenity - missing likely means absent)

PoolPrivateYN: NMAR (amenity - missing likely means absent)

AttachedGarageYN: NMAR (amenity - missing likely means absent)

LotSizeAcres: MAR detected (correlates with 1 features)
  - Latitude: r=-0.136


c:\Users\jueeh\idx-exchange-ds34\venv\Lib\site-packages\numpy\lib\_function_base_impl.py:3065: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
c:\Users\jueeh\idx-exchange-ds34\venv\Lib\site-packages\numpy\lib\_function_base_impl.py:3066: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
c:\Users\jueeh\idx-exchange-ds34\venv\Lib\site-packages\numpy\lib\_function_base_impl.py:3065: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
c:\Users\jueeh\idx-exchange-ds34\venv\Lib\site-packages\numpy\lib\_function_base_impl.py:3066: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]



FireplaceYN: NMAR (amenity - missing likely means absent)

Stories: MAR detected (correlates with 2 features)
  - Latitude: r=0.298
  - Longitude: r=-0.218

LotSizeArea: MAR detected (correlates with 1 features)
  - Latitude: r=-0.136

GarageSpaces: MAR detected (correlates with 1 features)
  - YearBuilt: r=-0.148

LotSizeSquareFeet: MAR detected (correlates with 2 features)
  - Latitude: r=-0.135
  - LotSizeAcres: r=0.874

--------------------------------------------------------------------------------
MECHANISM SUMMARY:
  MCAR: 11 columns
  MAR:  5 columns
  NMAR: 4 columns

ADAPTIVE IMPUTATION

ViewYN (NMAR): Created missing indicator, filled with False

PoolPrivateYN (NMAR): Created missing indicator, filled with False

Latitude (MCAR): Filled with 34.08

Longitude (MCAR): Filled with -118.03

LivingArea (MCAR): Filled with 1812.00

AttachedGarageYN (NMAR): Created missing indicator, filled with False

ParkingTotal (MCAR): Filled with 0.00

LotSizeAcres (MAR): Will use model-based

c:\Users\jueeh\idx-exchange-ds34\venv\Lib\site-packages\numpy\lib\_function_base_impl.py:3065: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
c:\Users\jueeh\idx-exchange-ds34\venv\Lib\site-packages\numpy\lib\_function_base_impl.py:3066: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
c:\Users\jueeh\idx-exchange-ds34\xgboost_juee\cleaning_script.py:204: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[col] = df[col].fillna(False)
c:\Users\jueeh\idx-exchange-ds34\xgboost_juee\cleaning_script.py:204: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_opti


FireplaceYN (NMAR): Created missing indicator, filled with False

Stories (MAR): Will use model-based imputation

LotSizeArea (MAR): Will use model-based imputation

GarageSpaces (MAR): Will use model-based imputation

LotSizeSquareFeet (MAR): Will use model-based imputation

--------------------------------------------------------------------------------
Applying IterativeImputer to 5 MAR numeric features...
  ✓ Fitted IterativeImputer

⚠️  Remaining nulls: {'City': 58, 'PurchaseContractDate': 5, 'PostalCode': 2, 'SourceFile': 34360}

CLEANING COMPLETE
Final shape: (66871, 31)
Rows retained: 66871/101297 (66.0%)

Missingness indicators created: 4
  - ViewYN_was_missing: 9.2% of data
  - PoolPrivateYN_was_missing: 7.9% of data
  - AttachedGarageYN_was_missing: 11.9% of data
  - FireplaceYN_was_missing: 0.1% of data
ADAPTIVE CLEANING - TEST DATA
Initial shape: (23233, 78)

Dropping 20 columns with >75% missing

ADAPTIVE IMPUTATION

ViewYN (NMAR): Created missing indicator, filled with 

c:\Users\jueeh\idx-exchange-ds34\xgboost_juee\cleaning_script.py:204: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[col] = df[col].fillna(False)
c:\Users\jueeh\idx-exchange-ds34\xgboost_juee\cleaning_script.py:204: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[col] = df[col].fillna(False)
c:\Users\jueeh\idx-exchange-ds34\xgboost_juee\cleaning_script.py:204: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the futur

GOAL: BRAINSTORM NEIGHBORHOOD LEVEL FEATURES

In [50]:
train_cleaned.columns
location_cols = ['Latitude', 'Longitude', 'CountyOrParish', 'City', 'PostalCode']

In [52]:
train_cleaned[['PostalCode', 'ClosePrice']]
train_cleaned['PostalCode'].value_counts()

PostalCode
92253         557
92223         396
92584         393
92345         390
92592         351
             ... 
94509-5872      1
94513-6500      1
94513-2620      1
94519-1620      1
93528           1
Name: count, Length: 1652, dtype: int64